**전처리: 지하철역 - 따릉이 대여소 근접 관계 테이블 생성**

* 수집 단계에서 DB에 저장한 **원본 테이블을 불러와서** 전처리 후 새 테이블을 만드는 노트북 (수집 코드와 별개로 실행 가능)
* 사용하는 원본 테이블
  * `RAW_SUBWAY_STATION_MASTER` : 서울시 역사마스터 (역ID·역명·호선·위도·경도)
  * `RAW_BIKE_STATION` : 따릉이 대여소 정보 (대여소ID·이름·위도·경도)
* 만들 테이블 : `STATION_SUBWAY_LINK` (반경 안의 대여소-지하철역 쌍)
* **지도 API는 사용하지 않음** : 두 테이블 모두 위도·경도가 있으므로 직선거리(haversine)로 계산
  * 지도 API가 필요한 경우 : 좌표가 없는 대여소를 주소로 변환(지오코딩)하거나, 도보 경로 거리가 필요할 때
* 근처 기준 : 300m 이내 / 300~500m / 500m 초과는 연결하지 않음

#01\. 기본설치

In [ ]:
# [DB 연결 1] postgresql 연결 라이브러리 설치
%pip install -q psycopg2-binary

In [ ]:
# [DB 연결 2] 라이브러리
import getpass
import io
import re
import numpy as np
import pandas as pd
import psycopg2
from IPython.display import display

In [ ]:
# [DB 연결 3] 접속 정보 입력 후 연결 (수집 노트북과 같은 DB)
host   = "여기에_호스트"      # ← 수집 노트북에서 사용한 값과 동일하게
port   = 5432                  # ← 수집 노트북에서 사용한 값과 동일하게
dbname = "bicycle_db"
user   = "여기에_사용자"
pw     = getpass.getpass("비밀번호: ")


def _connect():
    return psycopg2.connect(
        host=host, port=port, dbname=dbname, user=user, password=pw,
        connect_timeout=15,
        keepalives=1, keepalives_idle=30,          # 연결 유지용 TCP keepalive
        keepalives_interval=10, keepalives_count=5,
    )

conn = _connect()


def ensure_conn():
    """연결이 끊겼으면 재접속"""
    global conn
    try:
        if conn.closed:
            raise psycopg2.InterfaceError("closed")
        with conn.cursor() as cur:
            cur.execute("SELECT 1")
        conn.rollback()                             # 확인용 트랜잭션 정리
    except Exception:
        try:
            conn.close()
        except Exception:
            pass
        conn = _connect()
        print("[재접속 완료]")

with conn.cursor() as cur:
    cur.execute("SELECT current_user, current_database(), version()")
    print("접속 성공:", cur.fetchone())
conn.rollback()

#02\. 원본 테이블 불러오기

In [ ]:
# ============================================================
# [설정값 + 공통 함수]
#   - 대여소 정보(반기) 대신 대여소 마스터를 쓰려면 BIKE_RAW_TABLE 만 바꾸면 됨
# ============================================================
SUBWAY_RAW_TABLE = "raw_subway_station_master"      # 역사마스터 원본 테이블
BIKE_RAW_TABLE   = "raw_bike_station"               # 대여소 정보 원본 테이블 (또는 raw_bike_station_master)
LINK_TABLE       = "STATION_SUBWAY_LINK"            # 새로 만들 결과 테이블

NEAR_RADIUS_M     = 300     # 가까움 구간 경계
MAX_LINK_RADIUS_M = 500     # 연결 최대 반경
SAME_STATION_M    = 700     # 같은 이름의 역이 이 거리 이상 벌어지면 다른 역으로 분리

MANUAL_NAME_MAP = {}        # 역명 표기가 다를 때 수동 매핑 예) {"총신대입구(이수)": "이수"}


def table_exists(table_name):
    with conn.cursor() as cur:
        cur.execute("SELECT to_regclass(%s::text)", [table_name.lower()])
        exists = cur.fetchone()[0] is not None
    conn.rollback()
    return exists


def read_table(table_name):
    """테이블 전체를 DataFrame 으로 불러오기 (원본 테이블은 전부 TEXT)"""
    if not table_exists(table_name):
        raise RuntimeError(f"{table_name} 테이블이 없습니다. 수집 단계에서 저장됐는지 확인하세요.")
    ensure_conn()
    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {table_name}")
        df = pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
    conn.rollback()
    return df


def pick_col(columns, candidates, label):
    """후보 이름 중 실제 존재하는 컬럼명 반환 (대소문자 무시)"""
    lookup = {str(c).strip().upper(): c for c in columns}
    for cand in candidates:
        if cand.upper() in lookup:
            return lookup[cand.upper()]
    raise ValueError(f"[{label}] 컬럼을 찾지 못했습니다. 후보: {candidates}\n  현재 컬럼: {list(columns)}")


def haversine_m(lat1, lon1, lat2, lon2):
    """두 좌표 사이 직선거리(m). numpy 배열 브로드캐스팅 지원"""
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = (np.sin((lat2 - lat1) / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    return 6371008.8 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def normalize_name(name):
    """역명 정규화: 수동 매핑 → 괄호 내용 제거 → 공백 제거 → 끝의 '역' 제거"""
    s = str(name).strip()
    s = MANUAL_NAME_MAP.get(s, s)
    s = re.sub(r"\(.*?\)", "", s)
    s = re.sub(r"\s+", "", s)
    return re.sub(r"역$", "", s)


def normalize_line(line):
    """호선 정규화: '01호선', '1호선', '1' → '1호선' / 그 외 노선명은 공백만 제거"""
    s = str(line).strip()
    m = re.fullmatch(r"0*(\d+)\s*(호선)?", s)
    return f"{int(m.group(1))}호선" if m else re.sub(r"\s+", "", s)


def fix_coords(df, lat_col, lon_col):
    """좌표를 숫자로 바꾸고, 위경도가 뒤바뀐 행은 교정, 범위 밖/결측 행은 오류 표로 분리"""
    out = df.copy()
    out[lat_col] = pd.to_numeric(out[lat_col], errors="coerce")
    out[lon_col] = pd.to_numeric(out[lon_col], errors="coerce")
    swapped = out[lat_col].between(124, 132) & out[lon_col].between(33, 39)
    out.loc[swapped, [lat_col, lon_col]] = out.loc[swapped, [lon_col, lat_col]].to_numpy()
    valid = out[lat_col].between(37.0, 38.2) & out[lon_col].between(126.0, 128.0)
    return out[valid].copy(), out[~valid].copy(), int(swapped.sum())

In [ ]:
# ============================================================
# [테이블 불러오기] 원본 테이블 → DataFrame, 컬럼 / 행 수 / 샘플 확인
# ============================================================
df_subway_raw = read_table(SUBWAY_RAW_TABLE)
df_bike_raw   = read_table(BIKE_RAW_TABLE)

for name, df in {SUBWAY_RAW_TABLE: df_subway_raw, BIKE_RAW_TABLE: df_bike_raw}.items():
    print(f"\n===== {name} ({len(df):,}행) =====")
    print("컬럼:", df.columns.tolist())
    display(df.head(3))

In [ ]:
# ============================================================
# [컬럼 지정] 후보 이름에서 자동 탐지, 틀리면 직접 덮어쓰기
#   - 아래 출력된 매핑이 맞는지 반드시 확인
# ============================================================
SUBWAY_COLS = {
    "id":   pick_col(df_subway_raw.columns, ["BLDN_ID", "STATN_ID", "STATION_ID", "SBWY_STNS_ID"], "역ID"),
    "name": pick_col(df_subway_raw.columns, ["BLDN_NM", "STATN_NM", "STATION_NM", "SBWY_STNS_NM"], "역명"),
    "line": pick_col(df_subway_raw.columns, ["ROUTE", "LINE_NUM", "LINE_NM", "SBWY_ROUT_LN_NM"], "호선"),
    "lat":  pick_col(df_subway_raw.columns, ["LAT", "LATITUDE"], "위도"),
    "lon":  pick_col(df_subway_raw.columns, ["LOT", "LNG", "LON", "LONGITUDE"], "경도"),
}
BIKE_COLS = {
    "id":   pick_col(df_bike_raw.columns, ["RENT_ID", "STATION_ID", "STATN_ID", "STA_ID", "RENT_NO"], "대여소ID"),
    "name": pick_col(df_bike_raw.columns, ["RENT_NM", "STATION_NAME", "STATN_NM", "RENT_ID_NM", "STA_NM"], "대여소명"),
    "lat":  pick_col(df_bike_raw.columns, ["STA_LAT", "LATITUDE", "LAT", "STATN_LAT"], "위도"),
    "lon":  pick_col(df_bike_raw.columns, ["STA_LONG", "STA_LNG", "LONGITUDE", "LON", "LNG", "STATN_LNT", "STATN_LOT"], "경도"),
}
# 자동 탐지가 틀렸다면 직접 지정 → 예) BIKE_COLS["id"] = "rent_id"

print("지하철 컬럼 매핑:", SUBWAY_COLS)
print("대여소 컬럼 매핑:", BIKE_COLS)

#03\. 전처리

In [ ]:
# ============================================================
# [전처리 1] 지하철역 : 좌표 검증 → 호선별 역(subway_points) → 물리역(subway_stations)
#   - 위경도가 뒤바뀐 행은 교정, 수도권 범위 밖/결측/0 좌표는 오류 표로 분리
#   - 환승역은 같은 위치의 역 1곳(물리역)으로 묶음
#   - 이름만 같고 SAME_STATION_M 이상 떨어진 역(예: 5호선 양평 / 경의중앙선 양평)은 호선별로 분리
# ============================================================
sub_ok, sub_err, sub_swapped = fix_coords(df_subway_raw, SUBWAY_COLS["lat"], SUBWAY_COLS["lon"])

subway_points = pd.DataFrame({
    "subway_id":   sub_ok[SUBWAY_COLS["id"]].astype("string").str.strip(),
    "subway_name": sub_ok[SUBWAY_COLS["name"]].astype("string").str.strip(),
    "line":        sub_ok[SUBWAY_COLS["line"]].astype("string").str.strip(),
    "lat":         sub_ok[SUBWAY_COLS["lat"]],
    "lon":         sub_ok[SUBWAY_COLS["lon"]],
}).drop_duplicates(subset=["subway_id"]).reset_index(drop=True)
subway_points["name_key"] = subway_points["subway_name"].map(normalize_name)
subway_points["line_key"] = subway_points["line"].map(normalize_line)

subway_points["phys_key"] = subway_points["name_key"]
for name_key, g in subway_points.groupby("name_key"):
    spread = haversine_m(g["lat"], g["lon"], g["lat"].mean(), g["lon"].mean()).max()
    if spread > SAME_STATION_M:
        subway_points.loc[g.index, "phys_key"] = name_key + "_" + g["line_key"]

subway_stations = (
    subway_points.groupby("phys_key")
    .agg(subway_name=("subway_name", "first"),
         lat=("lat", "mean"), lon=("lon", "mean"),
         lines=("line_key", lambda s: ", ".join(sorted(s.unique()))),
         n_lines=("line_key", "nunique"))
    .reset_index()
)

print(f"[역] 원본 {len(df_subway_raw):,}행 → 유효 {len(subway_points):,}행 / 좌표 오류 {len(sub_err)}행 / 위경도 뒤바뀜 교정 {sub_swapped}행")
print(f"[물리역] {len(subway_stations):,}곳 / 환승역(2개 호선 이상) {(subway_stations['n_lines'] >= 2).sum():,}곳")
if len(sub_err):
    print("\n좌표 오류 역:"); display(sub_err.head(20))

In [ ]:
# ============================================================
# [전처리 2] 따릉이 대여소 : 좌표 검증 + ID 중복 제거
# ============================================================
bike_ok, bike_err, bike_swapped = fix_coords(df_bike_raw, BIKE_COLS["lat"], BIKE_COLS["lon"])

bike_stations = pd.DataFrame({
    "station_id":   bike_ok[BIKE_COLS["id"]].astype("string").str.strip(),
    "station_name": bike_ok[BIKE_COLS["name"]].astype("string").str.strip(),
    "latitude":     bike_ok[BIKE_COLS["lat"]],
    "longitude":    bike_ok[BIKE_COLS["lon"]],
})
n_dup = int(bike_stations.duplicated("station_id").sum())
bike_stations = bike_stations.drop_duplicates("station_id").reset_index(drop=True)

print(f"[대여소] 원본 {len(df_bike_raw):,}행 → 유효 {len(bike_stations):,}행 / 좌표 오류 {len(bike_err)}행 / "
      f"위경도 뒤바뀜 교정 {bike_swapped}행 / ID 중복 제거 {n_dup}행")
if len(bike_err):
    print("\n좌표 오류 대여소:"); display(bike_err.head(20))

In [ ]:
# ============================================================
# [전처리 3] 거리 계산 → 근접 관계 DataFrame (station_subway_link)
#   - 대여소 × 물리역 직선거리 행렬(numpy) → 반경 500m 안의 쌍만 남김
# ============================================================
dist = haversine_m(
    bike_stations["latitude"].to_numpy()[:, None],
    bike_stations["longitude"].to_numpy()[:, None],
    subway_stations["lat"].to_numpy()[None, :],
    subway_stations["lon"].to_numpy()[None, :],
)                                                   # shape = (대여소 수, 물리역 수)

bi, si = np.where(dist <= MAX_LINK_RADIUS_M)
station_subway_link = pd.DataFrame({
    "station_id":   bike_stations["station_id"].to_numpy()[bi],
    "station_name": bike_stations["station_name"].to_numpy()[bi],
    "phys_key":     subway_stations["phys_key"].to_numpy()[si],
    "subway_name":  subway_stations["subway_name"].to_numpy()[si],
    "lines":        subway_stations["lines"].to_numpy()[si],
    "distance_m":   dist[bi, si].round(1),
})
station_subway_link["radius_band"] = np.where(
    station_subway_link["distance_m"] <= NEAR_RADIUS_M,
    f"0-{NEAR_RADIUS_M}m", f"{NEAR_RADIUS_M}-{MAX_LINK_RADIUS_M}m")
# 한 대여소가 여러 역 반경에 걸칠 때 가까운 순서
station_subway_link["rank_in_station"] = (
    station_subway_link.groupby("station_id")["distance_m"].rank(method="first").astype(int))
station_subway_link["is_nearest"] = station_subway_link["rank_in_station"].eq(1)
station_subway_link["radius_m"] = MAX_LINK_RADIUS_M
station_subway_link = station_subway_link.sort_values(["station_id", "rank_in_station"]).reset_index(drop=True)

# 반경 밖 대여소도 포함한 '가장 가까운 역까지 거리' (확인용)
nearest_m = pd.Series(dist.min(axis=1), index=bike_stations["station_id"])
print(f"연결 쌍: {len(station_subway_link):,}건 / 역 근처 대여소: {station_subway_link['station_id'].nunique():,}곳 "
      f"/ 근처 대여소가 있는 역: {station_subway_link['phys_key'].nunique():,}곳")
display(station_subway_link.head(10))

In [ ]:
# ============================================================
# [기준 확인] 거리 분포, 반경 밖 대여소, 상식 점검
# ============================================================
for r in (NEAR_RADIUS_M, MAX_LINK_RADIUS_M):
    print(f"가장 가까운 역이 {r}m 이내인 대여소: {(nearest_m <= r).mean():.1%}")
print(f"반경 {MAX_LINK_RADIUS_M}m 안에 역이 없는 대여소: {(nearest_m > MAX_LINK_RADIUS_M).sum():,}곳 / {len(bike_stations):,}곳")

print("\n가장 가까운 역까지 거리(m) 분포")
display(nearest_m.describe().round(0).to_frame("nearest_m"))
print("역당 근처 대여소 수 분포")
display(station_subway_link.groupby("phys_key").size().describe().round(1).to_frame("대여소 수"))

# 좌표가 한쪽만 틀어진 경우 모든 대여소가 '반경 밖'으로 나옴 → 중심 간 거리로 점검
gap_km = haversine_m(bike_stations["latitude"].mean(), bike_stations["longitude"].mean(),
                     subway_stations["lat"].mean(), subway_stations["lon"].mean()) / 1000
print(f"\n대여소 중심 ↔ 역 중심 거리: {gap_km:.1f} km  (서울 안이면 수 km 이내가 정상)")
if (nearest_m > MAX_LINK_RADIUS_M).mean() > 0.9 or gap_km > 10:
    print("⚠ 거의 모든 대여소가 반경 밖입니다. 컬럼 매핑을 다시 확인하세요.")

check_name = "합정"          # 상식 점검 : 아는 역을 넣어 보기
print(f"\n'{check_name}' 근처 대여소 (가까운 순)")
display(station_subway_link[station_subway_link["subway_name"].str.contains(check_name, na=False)]
        .sort_values("distance_m").head(10))

#04\. 테이블 생성

In [ ]:
# ============================================================
# [테이블 생성 + 저장] STATION_SUBWAY_LINK
#   - 계산 결과(거리, 순위 등)는 TEXT 가 아닌 형식을 지정해서 저장
#   - 계산으로 언제든 다시 만들 수 있는 테이블이라 한 트랜잭션에서 비우고 다시 채움
# ============================================================
link_cols = ["station_id", "phys_key", "station_name", "subway_name", "lines",
             "distance_m", "radius_band", "rank_in_station", "is_nearest", "radius_m"]

if station_subway_link.empty:
    raise RuntimeError("저장할 근접 관계가 없습니다. 컬럼 매핑과 좌표 검증 결과를 먼저 확인하세요.")

ensure_conn()
t = LINK_TABLE.lower()
try:
    with conn.cursor() as cur:
        cur.execute(f"""CREATE TABLE IF NOT EXISTS {t} (
            station_id      TEXT             NOT NULL,   -- 따릉이 대여소
            phys_key        TEXT             NOT NULL,   -- 물리역 키 (환승역은 1개)
            station_name    TEXT,
            subway_name     TEXT,
            lines           TEXT,                        -- 예: '2호선, 6호선'
            distance_m      DOUBLE PRECISION NOT NULL,   -- 직선거리(m)
            radius_band     TEXT,                        -- '0-300m' / '300-500m'
            rank_in_station INTEGER,                     -- 그 대여소에서 가까운 역 순서
            is_nearest      BOOLEAN,                     -- 그 대여소의 가장 가까운 역인지
            radius_m        INTEGER,                     -- 연결에 사용한 최대 반경
            built_at        TIMESTAMPTZ DEFAULT now(),
            PRIMARY KEY (station_id, phys_key))""")
        cur.execute(f"CREATE INDEX IF NOT EXISTS idx_{t}_phys ON {t} (phys_key)")
        cur.execute(f"TRUNCATE {t}")                      # 다시 실행해도 중복 없음

        buf = io.StringIO()
        station_subway_link[link_cols].to_csv(buf, index=False, header=False)
        buf.seek(0)
        cur.copy_expert(f"COPY {t} ({', '.join(link_cols)}) FROM STDIN WITH (FORMAT csv)", buf)
    conn.commit()
except Exception:
    conn.rollback()
    raise
print(f"{t.upper()}: {len(station_subway_link):,}행 저장 완료")

In [ ]:
# ============================================================
# [저장 결과 확인]
# ============================================================
t = LINK_TABLE.lower()
with conn.cursor() as cur:
    cur.execute(f"SELECT COUNT(*), COUNT(DISTINCT station_id), COUNT(DISTINCT phys_key), "
                f"SUM(is_nearest::int) FROM {t}")
    n, n_station, n_subway, n_nearest = cur.fetchone()
    print(f"행 {n:,} / 대여소 {n_station:,}곳 / 역 {n_subway:,}곳 / 가장 가까운 역 표시 {n_nearest:,}건")
    print("DataFrame 과 행 수:", "일치 ✔" if n == len(station_subway_link) else "불일치 ⚠")

    cur.execute(f"""SELECT radius_band, COUNT(*), ROUND(AVG(distance_m)::numeric, 1)
                    FROM {t} GROUP BY radius_band ORDER BY radius_band""")
    for band, cnt, avg in cur.fetchall():
        print(f"  {band}: {cnt:,}건 (평균 {avg}m)")
conn.rollback()